In [15]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tarnet import Tarnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/RERUM")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [16]:
%time train_df = pd.read_csv(r"../dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"../dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"../dataset/Hillstrom/Men/val_men.csv")

CPU times: user 22.9 ms, sys: 10 ms, total: 32.9 ms
Wall time: 31.8 ms
CPU times: user 11.5 ms, sys: 5.02 ms, total: 16.5 ms
Wall time: 16.1 ms
CPU times: user 5.22 ms, sys: 0 ns, total: 5.22 ms
Wall time: 5.21 ms


In [17]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [18]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [19]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [20]:
epochs = 150
early_stop_metric = "ema_qini"
ema = True
ema_alpha = 0.25
patience = 20
early_stop_start = 30
print (f" epochs = {epochs}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" early stop start = {early_stop_start}")

 epochs = 150
 early stop = ema_qini
 use ema = True
 ema alpha = 0.25
 patience = 20
 early stop start = 30


In [21]:
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 100
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    grid_outcome_dropout = trial.suggest_float("outcome_dropout", 0.1, 0.5)
    grid_shared_dropout = trial.suggest_float("shared_dropout", 0.0, 0.4    )
    grid_shared_hidden = trial.suggest_categorical("shared_hidden", [128, 256, 384, 512])
    grid_outcome_hidden = trial.suggest_categorical("outcome_hidden", [64, 128, 256])
    grid_ziln_lambda = trial.suggest_float("ziln_lambda", 5, 50.0, log=True)
    grid_pos_weight = trial.suggest_float("pos_weight", 1.0, 20.0, log=True)

    val_auqc_list = []
    val_ate_err_list = []
    for SEED in seeds:
        seed_everything(SEED)

        tarnet = Tarnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=grid_shared_hidden,
            outcome_hidden=grid_outcome_hidden,
            outcome_dropout=grid_outcome_dropout,
            shared_dropout=grid_shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            ziln_lambda=grid_ziln_lambda,
            pos_weight=grid_pos_weight
        )

        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            tarnet.fit(train_loader, val_loader)

        x_men_val_t_on_device = x_men_val_t.to(device)
        y0_pred, y1_pred = tarnet.predict(x_men_val_t_on_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_val_t.detach().cpu().numpy().flatten()
        t_true = t_men_val_t.detach().cpu().numpy().flatten()
        
        current_val_auqc = auqc(y_true, t_true, uplift_pred, bins=100, plot=False)
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
        current_val_ate_err = abs(ate_pred - ate_true)

        val_auqc_list.append(current_val_auqc)
        val_ate_err_list.append(current_val_ate_err)

    # Calculate aggregated metrics across the 5 validation seeds
    mean_auqc = float(np.mean(val_auqc_list))
    std_auqc = float(np.std(val_auqc_list))
    mean_ate_err = float(np.mean(val_ate_err_list))

    # Apply penalty for instability and miscalibration
    penalty_std = std_auqc * 0.5
    penalty_ate = mean_ate_err * 0.05

    final_score = mean_auqc - penalty_std - penalty_ate - penalty_std

    trial.set_user_attr("mean_val_auqc", mean_auqc)
    trial.set_user_attr("std_Val_auqc", std_auqc)
    trial.set_user_attr("mean_val_ate_err", mean_ate_err)
    trial.set_user_attr("final_score", final_score)
    return final_score

def print_trial_callback(study, trial):
    value = float(trial.value) if trial.value is not None else float("nan")
    best_trial = study.best_trial
    best_value = float(best_trial.value) if best_trial.value is not None else float("nan")
    print(
        f"Finished trial {trial.number}: val score: {value:.4f} - "
        f"with hyperparameters: {trial.params} | "
        f"best trial: {best_trial.number} score: {best_value:.4f}",
        flush=True
    )

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[print_trial_callback])

trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": round(float(t.params["lr"]), 4),
        "weight_decay": round(float(t.params["weight_decay"]), 4),
        "shared_hidden": int(t.params["shared_hidden"]),
        "outcome_hidden": int(t.params["outcome_hidden"]),
        "shared_dropout": round(float(t.params["shared_dropout"]), 3),
        "outcome_dropout": round(float(t.params["outcome_dropout"]), 3),
        "ziln_lambda": round(float(t.params["ziln_lambda"]), 3),
        "pos_weight": round(float(t.params["pos_weight"]), 3),
        "mean_val_auqc": float(t.value),
        "std_Val_auqc": float(t.user_attrs.get("std_Val_auqc", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_val_auqc", ascending=True).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "shared_hidden": int(best_params["shared_hidden"]),
    "outcome_hidden": int(best_params["outcome_hidden"]),
    "shared_dropout": float(best_params["shared_dropout"]),
    "outcome_dropout": float(best_params["outcome_dropout"]),
    "ziln_lambda": float(best_params["ziln_lambda"]),
    "pos_weight": float(best_params["pos_weight"]),
    "mean_Val_auqc": float(study.best_value),
    "std_Val_auqc": float(study.best_trial.user_attrs.get("std_Val_auqc", np.nan))
})

  0%|          | 0/100 [00:00<?, ?it/s]

Finished trial 0: val score: 0.5050 - with hyperparameters: {'lr': 9.008296332262975e-05, 'weight_decay': 3.2543813547868006e-05, 'outcome_dropout': 0.26330970370677254, 'shared_dropout': 0.17735402643744136, 'shared_hidden': 512, 'outcome_hidden': 256, 'ziln_lambda': 10.235980761081278, 'pos_weight': 3.4889116308840054} | best trial: 0 score: 0.5050


Best trial: 0. Best value: 0.504956:   1%|          | 1/100 [01:18<2:09:31, 78.50s/it]

Finished trial 1: val score: 0.4617 - with hyperparameters: {'lr': 2.2049986176238194e-05, 'weight_decay': 7.293306303133679e-06, 'outcome_dropout': 0.3671615491304012, 'shared_dropout': 0.30697964005463274, 'shared_hidden': 128, 'outcome_hidden': 256, 'ziln_lambda': 29.735895976238673, 'pos_weight': 2.8747396589909275} | best trial: 0 score: 0.5050


Best trial: 0. Best value: 0.504956:   2%|▏         | 2/100 [03:10<2:40:07, 98.04s/it]

...

Best trial: 53. Best value: 0.615044:  98%|█████████▊| 98/100 [2:27:10<02:39, 79.83s/it]

Finished trial 98: val score: 0.5180 - with hyperparameters: {'lr': 1.2426861008552955e-05, 'weight_decay': 3.932587071008923e-05, 'outcome_dropout': 0.3025639802028496, 'shared_dropout': 0.22824452660378744, 'shared_hidden': 384, 'outcome_hidden': 256, 'ziln_lambda': 34.198815498584935, 'pos_weight': 5.179233580889123} | best trial: 53 score: 0.6150


Best trial: 53. Best value: 0.615044:  99%|█████████▉| 99/100 [2:28:59<01:28, 88.80s/it]

Finished trial 99: val score: 0.6151 - with hyperparameters: {'lr': 1.0854831006299919e-05, 'weight_decay': 8.908710439814211e-06, 'outcome_dropout': 0.3234603151576131, 'shared_dropout': 0.27800315821960286, 'shared_hidden': 384, 'outcome_hidden': 128, 'ziln_lambda': 31.34196044665029, 'pos_weight': 4.577124939815833} | best trial: 99 score: 0.6151


Best trial: 99. Best value: 0.615147: 100%|██████████| 100/100 [2:30:22<00:00, 90.23s/it]


In [22]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_Loss: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 99
Best mean_Val_Loss: 0.615147
Best params: {'lr': 1.0854831006299919e-05, 'weight_decay': 8.908710439814211e-06, 'outcome_dropout': 0.3234603151576131, 'shared_dropout': 0.27800315821960286, 'shared_hidden': 384, 'outcome_hidden': 128, 'ziln_lambda': 31.34196044665029, 'pos_weight': 4.577124939815833}

Best config table:


,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,ziln_lambda,pos_weight,mean_Val_auqc,std_Val_auqc
0,0.000011,0.000009,384.0,128.0,0.278003,0.32346,31.34196,4.577125,0.615147,0.051226



Top 10 trials:


,trial,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,ziln_lambda,pos_weight,mean_val_auqc,std_Val_auqc
0,47,0.0000,0.0000,128,64,0.237,0.387,31.901,14.721,0.196116,0.161331
1,39,0.0000,0.0000,128,64,0.226,0.215,22.396,6.410,0.197725,0.159989
2,4,0.0000,0.0000,128,64,0.139,0.348,28.629,13.891,0.202258,0.162450
3,68,0.0000,0.0001,128,128,0.190,0.452,14.734,15.549,0.326926,0.200931
4,2,0.0000,0.0007,128,128,0.077,0.413,16.001,7.124,0.327702,0.196000
5,87,0.0000,0.0000,128,128,0.261,0.279,29.977,14.260,0.332192,0.202858
6,6,0.0010,0.0001,384,256,0.125,0.149,13.548,5.246,0.341273,0.018150
7,20,0.0001,0.0002,512,128,0.268,0.242,6.150,1.005,0.373825,0.033948
8,59,0.0000,0.0004,256,64,0.259,0.340,45.964,8.708,0.376646,0.158061
9,17,0.0001,0.0011,384,128,0.006,0.107,6.280,1.353,0.408835,0.069173


In [23]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on test set (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
best_shared_hidden = int(best_cfg['shared_hidden'])
best_outcome_hidden = int(best_cfg['outcome_hidden'])
best_shared_dropout = float(best_cfg['shared_dropout'])
best_outcome_dropout = float(best_cfg['outcome_dropout'])
best_ziln_lambda = float(best_cfg['ziln_lambda'])
best_pos_weight = float(best_cfg['pos_weight'])

print("Evaluating on test with best validation config:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}")
print(f"  shared_hidden={best_shared_hidden}, outcome_hidden={best_outcome_hidden}")
print(f"  shared_dropout={best_shared_dropout:.3f}, outcome_dropout={best_outcome_dropout:.3f}")
print(f"Number of seeds: {len(seeds)}")

# 2. Loop over seeds for robust test evaluation
for SEED in seeds:
    seed_everything(SEED)

    tarnet = Tarnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=best_shared_hidden,
        outcome_hidden=best_outcome_hidden,
        outcome_dropout=best_outcome_dropout,
        shared_dropout=best_shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
        ziln_lambda=best_ziln_lambda,
        pos_weight=best_pos_weight
    )

    tarnet.fit(train_loader, val_loader)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = tarnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED}")

# 3. Aggregate final test metrics
df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'PER-SEED DETAILS (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'TEST SUMMARY (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Evaluating on test with best validation config:
  lr=1.1e-05, weight_decay=8.9e-06
  shared_hidden=384, outcome_hidden=128
  shared_dropout=0.278, outcome_dropout=0.323
Number of seeds: 5
🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: EMA_QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Best EMA Qini (alpha=0.25)
   Restore to epoch with highest smoothed (EMA) Qini score
   Patience: 20 epochs
Epoch 1/150 | Loss: 258.1118 | Cls: 2.3972 | Reg: 8.1589 | mu_t: 4.3789 | sigma_t: 0.7450 | mu_c: 4.3590 | sigma_c: 0.7277 | Val Loss: 9.2694 | F1_c: 0.0000 | PR_AUC_c: 0.0105 | F1_t: 0.0000 | PR_AUC_t: 0.0121 | Val Qini: 0.6895 | EMA Qini: 0.6895 | Best EMA: 0.6895 ⭐ NEW BEST EMA
Epoch 2/150 | Loss: 273.0853 | Cls: 2.3973 | Reg: 8.6366 | mu_t: 4.3790 | sigma_t: 0.7456 | mu_c: 4.3593 | sigma_c: 0.7282 | Val Loss: 9.2687 | F1_c: 0.0000 | PR_AUC_c: 0.0107 | F1_t: 0.0000 | PR_AUC_t: 0.0120 | Val Qini: 0.6979 | EMA Qini: 0.6916 | Best EMA: 0.6916 ⭐ NEW BEST EMA
Epoch 3/150 | Loss: 247.0017 | Cls: 2.39